In [1]:
import openai
from qdrant_client import QdrantClient

### Embedding function

In [2]:
def get_embedding(text,model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

### Retreival function

In [3]:
qdrant_client = QdrantClient(
    url="http://localhost:6333")

In [4]:
def retrieve_data(query,qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]
    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_context.append(point.payload["description"])
        similarity_scores.append(point.score)
        retrieved_context_ratings.append(point.payload["average_rating"])
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [6]:
retrieved_context = retrieve_data("What kind of earphones can I get?",qdrant_client, top_k=10)

In [7]:
retrieved_context

{'retrieved_context_ids': ['B0C61MMPVB',
  'B0BPRVVM66',
  'B0B4S2D5TB',
  'B0BPP86HGQ',
  'B0BNB4CRPS',
  'B0B12HQ7WJ',
  'B09QH2G56J',
  'B0B28JML9P',
  'B0B6ZNVP6R',
  'B0BJV9TZZ8'],
 'retrieved_context': ['Jicjocy Wireless Earbuds Bluetooth Active Noise Cancelling Earbuds, Hi-Fi Stereo Bluetooth Headphones with Charging Case, Waterproof in-Ear Earphones with Mic for iPhone/Android ♫【ANC Bluetooth Headphones】Bluetooth wireless earbuds have "Active Noise Cancellation" technology, which can cancel up to 40dB of noise, and ANC mode allows you to enjoy clear calls. You can also switch to Transparency mode at any time to hear your surroundings without removing the earbuds. ♫【Hi-Fi Stereo Sound】Bluetooth earbuds have built-in 13mm drivers and three-layer composite diaphragm provide deep bass, stunning highs and clear mids. Hi fi stereo sound earbuds are suitable for listening to music, making phone calls, watching videos and playing games. The natural and authentic sound for immersive mus

### Format retrieved context function

In [16]:
def process_context(context):
    formatted_context = ""
    for id, chunk,rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"-ID: {id}, Rating: {rating}, Description: {chunk}\n"

    return formatted_context

In [17]:
preprocessed_context = process_context(retrieved_context)

In [18]:
print(preprocessed_context)

-ID: B0C61MMPVB, Rating: 4.2, Description: Jicjocy Wireless Earbuds Bluetooth Active Noise Cancelling Earbuds, Hi-Fi Stereo Bluetooth Headphones with Charging Case, Waterproof in-Ear Earphones with Mic for iPhone/Android ♫【ANC Bluetooth Headphones】Bluetooth wireless earbuds have "Active Noise Cancellation" technology, which can cancel up to 40dB of noise, and ANC mode allows you to enjoy clear calls. You can also switch to Transparency mode at any time to hear your surroundings without removing the earbuds. ♫【Hi-Fi Stereo Sound】Bluetooth earbuds have built-in 13mm drivers and three-layer composite diaphragm provide deep bass, stunning highs and clear mids. Hi fi stereo sound earbuds are suitable for listening to music, making phone calls, watching videos and playing games. The natural and authentic sound for immersive music enjoyment. It can support the mono mode and twin stereo mode, you can share the earbuds with your friends and families. ♫【Auto Pairing, Smart Touch Control】Take out

### Create Prompt function

In [12]:
def build_prompt(preprocessed_context, question):
    prompt = f"""
you are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.

Context:
{preprocessed_context}

Question: {question}
"""    
    return prompt

In [19]:
prompt = build_prompt(preprocessed_context, "What kind of earphones can I get?")

In [20]:
print(prompt)


you are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.

Context:
-ID: B0C61MMPVB, Rating: 4.2, Description: Jicjocy Wireless Earbuds Bluetooth Active Noise Cancelling Earbuds, Hi-Fi Stereo Bluetooth Headphones with Charging Case, Waterproof in-Ear Earphones with Mic for iPhone/Android ♫【ANC Bluetooth Headphones】Bluetooth wireless earbuds have "Active Noise Cancellation" technology, which can cancel up to 40dB of noise, and ANC mode allows you to enjoy clear calls. You can also switch to Transparency mode at any time to hear your surroundings without removing the earbuds. ♫【Hi-Fi Stereo Sound】Bluetooth earbuds have built-in 13mm drivers and three-layer composite diaphragm provide deep bass, stunning highs and clear mids. Hi fi stereo sound earbuds are suitable 

### Generate answer function

In [21]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
                reasoning_effort="minimal"
        # temperature=0.7,
        # max_tokens=500,
    )
    return response.choices[0].message.content

In [22]:
print(generate_answer(prompt))

You can get wireless Bluetooth earphones with active noise cancellation (ANC) and charging case (example: Jicjocy Wireless Earbuds with ANC). They offer hi‑fi stereo sound, IPX8 waterproof coating, auto pairing, touch controls, and long battery life. If you prefer bone conduction/open-ear style, there are bone conduction headphones available as well (e.g., WATAHATIC Bone Conduction Headphones).


In [23]:
def rag_pipeline(question,top_k=5):
    qdrant_client = QdrantClient(
        url="http://localhost:6333")
    retrieved_context = retrieve_data(question,qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    return answer

In [25]:
print(rag_pipeline("What kind of earphones can I get with ratings above 4?",top_k=10))

You can get wireless earbuds with ANC, like the Jicjocy Wireless Earbuds (B0C61MMPVB) rated 4.2.
